In [ ]:
# STEP 1 — Select and prepare your dataset
from pathlib import Path
import zipfile, os, shutil

USE_GOOGLE_DRIVE = False

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    # Edit this to the ZIP file path in your Drive:
    ZIP_PATH = "/content/drive/MyDrive/UWI2020.zip"
    EXTRACT_DIR = Path("/content/dataset_upload")
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(EXTRACT_DIR)
else:
    from google.colab import files
    uploaded = files.upload()  # choose your dataset .zip
    zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
    if not zip_names:
        raise ValueError("Please upload a .zip file containing your dataset.")
    ZIP_PATH = "/content/" + zip_names[0]
    EXTRACT_DIR = Path("/content/dataset_upload")
    if EXTRACT_DIR.exists():
        shutil.rmtree(EXTRACT_DIR)
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(EXTRACT_DIR)

# Detect a folder with either:
#   dataset/train/class_name/... and optionally dataset/test/class_name/...
# or dataset/class_name/...
def looks_like_dataset(folder):
    folder = Path(folder)
    train_dir = folder / "train"
    if train_dir.is_dir():
        class_dirs = [p for p in train_dir.iterdir() if p.is_dir()]
        if len(class_dirs) >= 2:
            return True
    class_dirs = [p for p in folder.iterdir() if p.is_dir() and p.name.lower() not in {"train", "test", "valid", "val", "__macosx"}]
    image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    return len(class_dirs) >= 2 and any(
        f.is_file() and f.suffix.lower() in image_exts
        for d in class_dirs for f in d.rglob("*")
    )

candidates = [EXTRACT_DIR] + [p for p in EXTRACT_DIR.rglob("*") if p.is_dir()]
DATASET_ROOT = None
for candidate in candidates:
    if looks_like_dataset(candidate):
        DATASET_ROOT = str(candidate)
        break

if DATASET_ROOT is None:
    raise FileNotFoundError(
        "Could not automatically find the dataset folders. Expected train/class_name/image "
        "folders or class_name/image folders. Inspect /content/dataset_upload and set DATASET_ROOT manually."
    )

print("Dataset folder:", DATASET_ROOT)
print("Top-level items:", [p.name for p in Path(DATASET_ROOT).iterdir()])

In [ ]:
from pathlib import Path
import copy, json, random, time
import numpy as np
import pandas as pd
from PIL import Image, ImageEnhance, ImageFilter
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

# ---------------- CONFIG: edit this path ----------------
DATASET_DIR = Path(DATASET_ROOT)  # detected in the dataset setup cell above
OUT = Path("/content/hm_sa_whitebalance_robustness_results")
IMAGE_SIZE, BATCH_SIZE, EPOCHS = 224, 8, 30
LR, WEIGHT_DECAY = 1e-4, 1e-4
VAL_FRAC, TEST_FRAC, PATIENCE = 0.20, 0.15, 5
WORKERS, SEED, PRETRAINED, USE_AMP = 2, 42, True, True
# Supports: root/class_name/images OR root/train/class_name/images + root/test/class_name/images
# --------------------------------------------------------
OUT.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def seed_all():
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

In [ ]:
def gray_world(img):
    a = np.asarray(img.convert("RGB")).astype(np.float32)
    means = a.mean((0, 1)); target = means.mean()
    a *= (target / np.maximum(means, 1.0)).reshape(1, 1, 3)
    return Image.fromarray(np.clip(a, 0, 255).astype(np.uint8))

class WB:
    def __call__(self, img): return gray_world(img)

class Corruption:
    def __init__(self, kind, severity): self.kind, self.severity = kind, severity
    def __call__(self, img):
        img = img.convert("RGB"); s = self.severity
        if self.kind == "white_balance": return gray_world(img)
        if self.kind == "brightness":
            return ImageEnhance.Brightness(img).enhance({1:.70, 2:.85, 3:1.15, 4:1.30}[s])
        if self.kind == "contrast":
            return ImageEnhance.Contrast(img).enhance({1:.60, 2:.80, 3:1.20, 4:1.40}[s])
        if self.kind == "gaussian_blur":
            return img.filter(ImageFilter.GaussianBlur({1:.5, 2:1.0, 3:1.5, 4:2.0}[s]))
        if self.kind == "gaussian_noise":
            a = np.asarray(img).astype(np.float32)
            rng = np.random.default_rng(SEED + 100*s)
            a += rng.normal(0, {1:5, 2:10, 3:20, 4:30}[s], a.shape)
            return Image.fromarray(np.clip(a, 0, 255).astype(np.uint8))
        if self.kind == "color_shift":
            a = np.asarray(img).astype(np.float32)
            _, g, b = {1:(0,5,10), 2:(0,10,25), 3:(0,20,40), 4:(0,30,55)}[s]
            a[...,1] += g; a[...,2] += b
            return Image.fromarray(np.clip(a, 0, 255).astype(np.uint8))
        return img

class IndexedImages(Dataset):
    def __init__(self, base, indices, transform):
        self.base, self.indices, self.transform = base, list(indices), transform
    def __len__(self): return len(self.indices)
    def __getitem__(self, i):
        # base ImageFolder uses transform=None, so this returns a PIL image
        image, label = self.base[self.indices[i]]
        return self.transform(image), label

def transform_for(train=False, white_balance=False, corruption=None, severity=0):
    ops = [transforms.Resize((IMAGE_SIZE, IMAGE_SIZE))]
    if train: ops += [transforms.RandomHorizontalFlip(), transforms.RandomRotation(10)]
    if white_balance: ops += [WB()]
    if corruption: ops += [Corruption(corruption, severity)]
    ops += [transforms.ToTensor(),
            transforms.Normalize([.485,.456,.406], [.229,.224,.225])]
    return transforms.Compose(ops)

def get_data():
    tr_dir, te_dir = DATASET_DIR/"train", DATASET_DIR/"test"
    if tr_dir.is_dir():
        base = datasets.ImageFolder(str(tr_dir), transform=None)
        if len(base.classes) < 2:
            raise RuntimeError(f"Expected at least 2 class folders in {tr_dir}, found: {base.classes}")
        labels = np.array([y for _, y in base.samples])
        tr, va = train_test_split(np.arange(len(labels)), test_size=VAL_FRAC,
                                  random_state=SEED, stratify=labels)
        if te_dir.is_dir():
            test = datasets.ImageFolder(str(te_dir), transform=None)
            if test.classes != base.classes:
                raise ValueError(f"Class order mismatch: train={base.classes}; test={test.classes}")
            ti = np.arange(len(test))
        else:
            # No separate test folder: split train folder into train/val/test.
            tv, ti = train_test_split(np.arange(len(labels)), test_size=TEST_FRAC,
                                      random_state=SEED, stratify=labels)
            tr, va = train_test_split(tv, test_size=VAL_FRAC/(1-TEST_FRAC),
                                      random_state=SEED, stratify=labels[tv])
            test = base
        return base, tr, va, test, ti, base.classes
    base = datasets.ImageFolder(str(DATASET_DIR), transform=None)
    if len(base.classes) < 2: raise RuntimeError(f"Expected at least 2 class folders/images in {DATASET_DIR}; found {base.classes}")
    labels = np.array([y for _, y in base.samples])
    tv, ti = train_test_split(np.arange(len(labels)), test_size=TEST_FRAC,
                              random_state=SEED, stratify=labels)
    tr, va = train_test_split(tv, test_size=VAL_FRAC/(1-TEST_FRAC),
                              random_state=SEED, stratify=labels[tv])
    return base, tr, va, base, ti, base.classes

class HM_SANet(nn.Module):
    def __init__(self, n_classes, pretrained=True, ch=128):
        super().__init__()
        ew = models.EfficientNet_V2_S_Weights.DEFAULT if pretrained else None
        cw = models.ConvNeXt_Tiny_Weights.DEFAULT if pretrained else None
        self.eff = models.efficientnet_v2_s(weights=ew).features
        self.cnx = models.convnext_tiny(weights=cw).features
        self.ep = nn.Sequential(nn.Conv2d(1280,ch,1,bias=False),nn.BatchNorm2d(ch),nn.GELU())
        self.cp = nn.Sequential(nn.Conv2d(768,ch,1,bias=False),nn.BatchNorm2d(ch),nn.GELU())
        self.fuse = nn.Sequential(nn.Conv2d(ch*2,ch*2,3,padding=1,bias=False),
            nn.BatchNorm2d(ch*2),nn.GELU(),nn.Conv2d(ch*2,ch,1,bias=False),
            nn.BatchNorm2d(ch),nn.GELU())
        hidden=max(ch//16,1)
        self.mlp=nn.Sequential(nn.Conv2d(ch,hidden,1,bias=False),nn.ReLU(True),
                                nn.Conv2d(hidden,ch,1,bias=False))
        self.spatial=nn.Conv2d(2,1,7,padding=3,bias=False)
        self.refine=nn.Sequential(nn.Conv2d(ch,ch,3,padding=1,bias=False),
                                  nn.BatchNorm2d(ch),nn.GELU())
        self.pool=nn.AdaptiveAvgPool2d(1)
        self.head=nn.Sequential(nn.Flatten(),nn.Linear(ch,256),nn.GELU(),
                                nn.Dropout(.4),nn.Linear(256,n_classes))
    def forward(self,x):
        f=self.fuse(torch.cat([self.ep(self.eff(x)),self.cp(self.cnx(x))],1))
        ca=torch.sigmoid(self.mlp(f.mean((2,3),keepdim=True))+
                         self.mlp(torch.amax(f,(2,3),keepdim=True)))
        f=f*ca
        sa=torch.sigmoid(self.spatial(torch.cat([f.mean(1,keepdim=True),
                                                  torch.amax(f,1,keepdim=True)],1)))
        f=f*sa
        return self.head(self.pool(f+self.refine(f)))

In [ ]:
def loader(base, idx, tf, shuffle=False):
    return DataLoader(IndexedImages(base,idx,tf), batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=WORKERS, pin_memory=torch.cuda.is_available(),
                      persistent_workers=WORKERS > 0)

def epoch_run(model, dl, criterion, optimizer=None, scaler=None):
    train=optimizer is not None; model.train(train)
    loss_sum, ys, ps=0., [], []
    amp=USE_AMP and DEVICE.type=="cuda"
    for x,y in dl:
        x,y=x.to(DEVICE,non_blocking=True),y.to(DEVICE,non_blocking=True)
        if train: optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(train):
            with torch.autocast(device_type=DEVICE.type, enabled=amp):
                logits=model(x); loss=criterion(logits,y)
            if train:
                scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        loss_sum += loss.item()*len(y)
        ys.extend(y.cpu().tolist()); ps.extend(logits.argmax(1).cpu().tolist())
    return loss_sum/max(len(dl.dataset),1), np.array(ys), np.array(ps)

def metrics(y,p):
    pr,re,f1,_=precision_recall_fscore_support(y,p,average="macro",zero_division=0)
    return {"accuracy":accuracy_score(y,p),"macro_precision":pr,"macro_recall":re,"macro_f1":f1}

def train_variant(name, base, tr, va, classes):
    wb = name=="white_balance"
    train_dl=loader(base,tr,transform_for(train=True,white_balance=wb),True)
    val_dl=loader(base,va,transform_for(white_balance=wb))
    model=HM_SANet(len(classes),PRETRAINED).to(DEVICE)
    criterion=nn.CrossEntropyLoss()
    opt=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
    sched=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode="min",factor=.5,patience=2)
    scaler=torch.amp.GradScaler("cuda",enabled=USE_AMP and DEVICE.type=="cuda")
    best=float("inf"); best_state=None; wait=0; history=[]
    for ep in range(1,EPOCHS+1):
        tl,ty,tp=epoch_run(model,train_dl,criterion,opt,scaler)
        vl,vy,vp=epoch_run(model,val_dl,criterion)
        ta=accuracy_score(ty,tp); vaa=accuracy_score(vy,vp); sched.step(vl)
        history.append({"epoch":ep,"train_loss":tl,"train_accuracy":ta,
                        "val_loss":vl,"val_accuracy":vaa})
        print(f"{name} epoch {ep}/{EPOCHS}: train loss={tl:.4f} acc={ta:.4f}; val loss={vl:.4f} acc={vaa:.4f}")
        if vl<best:
            best=vl; wait=0; best_state=copy.deepcopy(model.state_dict())
            torch.save({"model_state_dict":best_state,"classes":classes,"variant":name},
                       OUT/f"hm_sa_net_{name}_best.pt")
        else:
            wait+=1
            if wait>=PATIENCE: print("Early stopping"); break
    model.load_state_dict(best_state)
    pd.DataFrame(history).to_csv(OUT/f"history_{name}.csv",index=False)
    fig,ax=plt.subplots(1,2,figsize=(10,4))
    h=pd.DataFrame(history)
    ax[0].plot(h.epoch,h.train_loss,label="train"); ax[0].plot(h.epoch,h.val_loss,label="validation")
    ax[0].set_title("Loss"); ax[0].legend()
    ax[1].plot(h.epoch,h.train_accuracy,label="train"); ax[1].plot(h.epoch,h.val_accuracy,label="validation")
    ax[1].set_title("Accuracy"); ax[1].legend()
    fig.tight_layout(); fig.savefig(OUT/f"curves_{name}.png",dpi=160); plt.close(fig)
    return model

@torch.no_grad()
def evaluate(model,test,ti,classes,condition="original",severity=0):
    tf=transform_for(corruption=None if condition=="original" else condition, severity=severity)
    dl=loader(test,ti,tf)
    _,y,p=epoch_run(model,dl,nn.CrossEntropyLoss())
    result=metrics(y,p); result.update({"condition":condition,"severity":severity})
    return result,y,p

In [ ]:
def main():
    seed_all()
    print("Device:",DEVICE)
    if torch.cuda.is_available():
        print("GPU:",torch.cuda.get_device_name(0),
              "VRAM GB:",round(torch.cuda.get_device_properties(0).total_memory/1024**3,2))
    base,tr,va,test,ti,classes=get_data()
    print("Classes:",classes, "| train:",len(tr),"validation:",len(va),"test:",len(ti))
    (OUT/"classes.json").write_text(json.dumps(classes,indent=2),encoding="utf-8")
    all_frames=[]
    for variant in ["original","white_balance"]:
        # Train separate models on original vs white-balanced training images.
        model=train_variant(variant,base,tr,va,classes)
        rows=[]
        # Both original and white-balanced test sets are evaluated for each model.
        for cond in ["original","white_balance"]:
            r,y,p=evaluate(model,test,ti,classes,cond,0)
            r["training_variant"]=variant; rows.append(r)
            if cond=="original":
                report=classification_report(y,p,labels=np.arange(len(classes)),
                    target_names=classes,output_dict=True,zero_division=0)
                pd.DataFrame(report).T.to_csv(OUT/f"report_{variant}_original_test.csv")
                cm=confusion_matrix(y,p,labels=np.arange(len(classes)))
                fig,ax=plt.subplots(figsize=(max(7,len(classes)*.55),max(6,len(classes)*.5)))
                im=ax.imshow(cm,interpolation="nearest"); fig.colorbar(im,ax=ax)
                ax.set_title(f"{variant} model - original test")
                ax.set_xlabel("Predicted"); ax.set_ylabel("True")
                ax.set_xticks(range(len(classes))); ax.set_xticklabels(classes,rotation=90)
                ax.set_yticks(range(len(classes))); ax.set_yticklabels(classes)
                fig.tight_layout(); fig.savefig(OUT/f"cm_{variant}.png",dpi=160); plt.close(fig)
        # Robustness: fixed checkpoint, no retraining on degraded test images.
        for cond in ["brightness","contrast","gaussian_noise","gaussian_blur","color_shift"]:
            for sev in [1,2,3,4]:
                r,_,_=evaluate(model,test,ti,classes,cond,sev)
                r["training_variant"]=variant; rows.append(r)
        df=pd.DataFrame(rows)
        base_row=df[(df.condition=="original")&(df.severity==0)].iloc[0]
        df["accuracy_change_pp"]=(df.accuracy-base_row.accuracy)*100
        df["macro_f1_change_pp"]=(df.macro_f1-base_row.macro_f1)*100
        df.to_csv(OUT/f"evaluation_{variant}.csv",index=False)
        all_frames.append(df)
        rob=df[df.condition.isin(["brightness","contrast","gaussian_noise","gaussian_blur","color_shift"])]
        for metric in ["accuracy","macro_f1"]:
            fig,ax=plt.subplots(figsize=(9,5))
            for cond in rob.condition.unique():
                s=rob[rob.condition==cond].sort_values("severity")
                ax.plot(s.severity,s[metric],marker="o",label=cond)
            ax.set_xlabel("Severity"); ax.set_ylabel(metric); ax.set_xticks([1,2,3,4])
            ax.set_title(f"{variant} model robustness - {metric}"); ax.legend()
            fig.tight_layout(); fig.savefig(OUT/f"robustness_{variant}_{metric}.png",dpi=160); plt.close(fig)
    all_df=pd.concat(all_frames,ignore_index=True)
    all_df.to_csv(OUT/"all_experiments_summary.csv",index=False)
    comp=all_df[(all_df.condition.isin(["original","white_balance"]))&(all_df.severity==0)]
    comp[["training_variant","condition","accuracy","macro_precision","macro_recall","macro_f1"]].to_csv(
        OUT/"white_balance_comparison.csv",index=False)
    print("\nWhite-balance comparison:")
    print(comp[["training_variant","condition","accuracy","macro_f1"]].to_string(index=False))
    print("\nFinished. Outputs saved to:",OUT.resolve())


main()

In [ ]:
import shutil
from google.colab import files
RESULTS_ZIP = shutil.make_archive(
    "/content/hm_sa_whitebalance_robustness_results",
    "zip",
    root_dir="/content/hm_sa_whitebalance_robustness_results"
)
print("Created:", RESULTS_ZIP)
files.download(RESULTS_ZIP)